# Spatial statistics {#sec-ind-spatial-statistics}



## Preamble

### Introduction

Spatial statistics builds around the first law of geography of @Tobler1970-urban-growth that states "everything is related to everything else, but near things are more related than distant things". This dependence of spatial observations has been studied in the field of (geo)spatial statistics. In general, spatial dependence is estimated by comparing the values at one location with the values at another location that is a given distance ('spatial lag') away [@Dale2014-book-spatial-analysis; @Baddeley2015-book-point-patterns].

The two technological streams, imaging- and sequencing-based assays, are very different in terms of data modalities: 

- In **imaging-based assays** (e.g., CosMx and Xenium) the observations of interest (e.g., mRNAs) are recorded where they occur natively. This means the locations of the observations are governed by a stochastic process and can be approximated as a **point process** or as an **irregular lattice**. 
- In **sequencing-based assays** (e.g., Visium), however, measurements are made along a defined grid, or **lattice**. This lattice is not created by a stochastic process and therefore cannot be approximated as a point process. Here, lattice data analysis methods have to be used. Notably, there are methods where cells can be segmented from very fine bins (e.g., Visium HD and IMC); these technologies can be approximated as being generated by a point process after segmentation [@Emons2025-pasta; @Baddeley2015-book-point-patterns; @Pebesma2023-book-spatial-data-science]. 

In the following vignette, the two main exploratory spatial statistics streams, point pattern analysis and lattice data analysis will be introduced.

### Dependencies

In [ ]:
library(dplyr)
library(scran)
library(spdep)
library(tidyr)
library(ggplot2)
library(Voyager)
library(SFEData)
library(spatstat)
library(openxlsx)
library(spatialFDA)
library(BiocParallel)
library(STexampleData)
library(SpatialExperiment)
library(SpatialFeatureExperiment)
# specify whether/how to 
# perform parallelization
bp <- MulticoreParam(4)
# set seed for random number generation
# in order to make results reproducible
set.seed(77)

In [ ]:
# load dataset as SPE & convert to SFE
spe <- Janesick_breastCancer_Xenium_rep1()
sfe <- toSpatialFeatureExperiment(spe)

# load the official 10X annotations 
fnm <- "https://cdn.10xgenomics.com/raw/upload/v1695234604/Xenium%20Preview%20Data/Cell_Barcode_Type_Matrices.xlsx"
labels <- read.xlsx(fnm, sheet=4)
labels$cell_id <- (labels$Barcode)

# add the cell type labels to the spe
cd <- as.data.frame(colData(sfe))
cd <- left_join(cd, 
    as.data.frame(labels), 
    by=join_by("cell_id"))
colData(sfe) <- DataFrame(cd)

# exclude 0-count cells &
# cells without annotation
sfe <- sfe[, colSums(counts(sfe)) > 0]
sfe <- sfe[, !is.na(sfe$Cluster)]

# log-library size normalization
sfe <- logNormCounts(sfe)

# basic theme for spatial plots
xy <- spatialCoords(sfe)
theme_xy <- list(
    coord_equal(expand=FALSE), 
    theme_void(), theme(
        plot.margin=margin(l=5),
        legend.key=element_blank(),
        panel.background=element_rect(fill="black")))

## Appendix

### References {.unnumbered}